**Pyspark Exercise 1**

In [44]:
!pip -q install pyspark

In [45]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .getOrCreate()
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)
stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()
sales_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2026-01-01")),
            (F.col("order_id") % 180).cast("int")
        )
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("P"),
            F.lpad(
                ((F.col("order_id") % 6) + 1).cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "store_id",
        F.concat(
            F.lit("S"),
            F.lpad(
                ((F.floor(F.col("order_id") / 3) % 4) + 1)
                .cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") % 8) + 1).cast("int")
    )
    .withColumn(
        "discount_pct",
        F.when(
            F.col("order_id") % 19 == 0,
            F.lit(None).cast("int")
        ).otherwise(
            ((F.floor(F.col("order_id") / 5) % 4) * 5)
            .cast("int")
        )
    )
    .withColumn(
        "order_status",
        F.when(F.col("order_id") % 11 == 0, "Cancelled")
         .when(F.col("order_id") % 7 == 0, "Returned")
         .otherwise("Completed")
    )

    .withColumn(
        "payment_method",
        F.when(F.col("order_id") % 3 == 0, "UPI")
         .when(F.col("order_id") % 3 == 1, "Card")
         .otherwise("Cash")
    )

    .withColumn(
        "salesperson_id",
        F.when(
            F.col("order_id") % 17 == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.concat(
                F.lit("EMP"),
                F.lpad(
                    ((F.col("order_id") % 12) + 1).cast("string"),
                    2, "0"
                )
            )
        )
    )

    .join(
        products_df.select(
            "product_id",
            F.col("list_price").alias("unit_price")
        ),
        "product_id",
        "left"
    )

    .select(
        "order_id", "order_date", "store_id",
        "product_id", "quantity", "unit_price",
        "discount_pct", "order_status",
        "payment_method", "salesperson_id"
    )
)

print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)

+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+

Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_p

In [46]:
sales_df.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price"
).show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       9|       P04|       2|     17000|
|      15|       P04|       8|     17000|
|      21|       P04|       6|     17000|
|       2|       P03|       3|      3500|
|       8|       P03|       1|      3500|
|      14|       P03|       7|      3500|
|      20|       P03|       5|      3500|
|       4|       P05|       5|      4500|
|      10|       P05|       3|      4500|
|      16|       P05|       1|      4500|
|       5|       P06|       6|     24000|
|      11|       P06|       4|     24000|
|      17|       P06|       2|     24000|
|       1|       P02|       2|     14000|
|       7|       P02|       8|     14000|
|      13|       P02|       6|     14000|
|      19|       P02|       4|     14000|
|       6|       P01|       7|     60000|
|      12|       P01|       5|     60000|
+--------+----------+--------+----

In [47]:
sales_df.filter(
    (F.col("order_status") == "Completed") &
    (F.col("quantity") > 5)
).show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      69|2026-03-11|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      38|2026-02-08|     S01|       P03|       7|      3500|        NULL|   Completed|          Cash|         EMP03|
|      62|2026-03-04|     S01|       P03|       7|      

In [48]:
sales_df = sales_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
)

sales_df.show(10)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|        68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|        34000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|        10500|
|       8|2026-01-09|     S03|       P03|       1|      3500|           5|   Completed|          Cash|         EMP09|         3500|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|  

In [49]:
sales_df = sales_df.withColumnRenamed(
    "order_status",
    "transaction_status"
).drop("payment_method")

sales_df.show(10)

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|        68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|         EMP10|        34000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|         EMP03|        10500|
|       8|2026-01-09|     S03|       P03|       1|      3500|           5|         Completed|         EMP09|         3500|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|         Completed|         EMP05|        22500|
|      10|2026-0

In [50]:
sales_df.groupBy("product_id").agg(
    F.sum("quantity").alias("total_quantity_sold")
).orderBy("product_id").show()

+----------+-------------------+
|product_id|total_quantity_sold|
+----------+-------------------+
|       P01|              66668|
|       P02|              83336|
|       P03|              66667|
|       P04|              83334|
|       P05|              66665|
|       P06|              83330|
+----------+-------------------+



In [51]:
category_revenue = (
    sales_df
    .join(products_df, "product_id", "inner")
    .withColumn(
        "gross_revenue",
        F.col("quantity") * F.col("list_price")
    )
    .groupBy("category")
    .agg(
        F.sum("gross_revenue").alias("total_gross_revenue")
    )
    .orderBy(F.desc("total_gross_revenue"))
)

category_revenue.show()

+-----------+-------------------+
|   category|total_gross_revenue|
+-----------+-------------------+
|Electronics|         7166704000|
|     Office|         1416678000|
| Networking|          299992500|
|Accessories|          233334500|
+-----------+-------------------+



In [52]:
city_transactions = (
    sales_df
    .join(stores_df, "store_id", "inner")
    .groupBy("city")
    .agg(
        F.count("*").alias("total_transactions")
    )
    .orderBy(F.desc("total_transactions"))
)

city_transactions.show()

+---------+------------------+
|     city|total_transactions|
+---------+------------------+
|Bengaluru|             25001|
|Hyderabad|             25001|
|   Mumbai|             24999|
|    Delhi|             24999|
+---------+------------------+



In [53]:
top_products = (
    sales_df
    .filter(F.col("order_status") == "Completed")
    .join(products_df, "product_id", "inner")
    .join(stores_df, "store_id", "inner")
    .withColumn(
        "gross_revenue",
        F.col("quantity") * F.col("list_price")
    )
    .groupBy("product_id", "product_name")
    .agg(
        F.sum("gross_revenue").alias("total_gross_revenue")
    )
    .orderBy(F.desc("total_gross_revenue"))
    .limit(5)
)

top_products.show()

+----------+------------+-------------------+
|product_id|product_name|total_gross_revenue|
+----------+------------+-------------------+
|       P01|      Laptop|         3116940000|
|       P06|      Tablet|         1558416000|
|       P04|     Printer|         1103912000|
|       P02|     Monitor|          909076000|
|       P05|      Router|          233766000|
+----------+------------+-------------------+



In [54]:
sales_df.select(
    F.sum(
        F.when(F.col("discount_pct").isNull(), 1).otherwise(0)
    ).alias("null_discount_count"),
    F.sum(
        F.when(F.col("salesperson_id").isNull(), 1).otherwise(0)
    ).alias("null_salesperson_count")
).show()

+-------------------+----------------------+
|null_discount_count|null_salesperson_count|
+-------------------+----------------------+
|               5263|                  5882|
+-------------------+----------------------+



In [55]:
sales_df = sales_df.na.fill({
    "discount_pct": 0
})

sales_df.select("discount_pct").show(10)

+------------+
|discount_pct|
+------------+
|           0|
|           5|
|           0|
|           5|
|           0|
|          10|
|           5|
|          10|
|           0|
|           5|
+------------+
only showing top 10 rows


In [56]:
sales_df = sales_df.na.drop(
    subset=["salesperson_id"]
)

print("Records after removing null salesperson IDs:", sales_df.count())

Records after removing null salesperson IDs: 94118


In [57]:
sales_df = sales_df.withColumn(
    "order_size",
    F.when(F.col("quantity") <= 3, "Small")
     .when(F.col("quantity") <= 6, "Medium")
     .otherwise("Large")
)

sales_df.select(
    "order_id", "quantity", "order_size"
).show(10)

+--------+--------+----------+
|order_id|quantity|order_size|
+--------+--------+----------+
|       3|       4|    Medium|
|       9|       2|     Small|
|       2|       3|     Small|
|       8|       1|     Small|
|       4|       5|    Medium|
|      10|       3|     Small|
|       5|       6|    Medium|
|      11|       4|    Medium|
|       1|       2|     Small|
|       7|       8|     Large|
+--------+--------+----------+
only showing top 10 rows


In [58]:
product_revenue = (
    sales_df
    .join(products_df, "product_id", "inner")
    .withColumn(
        "gross_revenue",
        F.col("quantity") * F.col("list_price")
    )
    .groupBy("category", "product_id", "product_name")
    .agg(
        F.sum("gross_revenue").alias("total_gross_revenue")
    )
)

category_window = Window.partitionBy("category").orderBy(
    F.desc("total_gross_revenue")
)

product_revenue = product_revenue.withColumn(
    "revenue_rank",
    F.dense_rank().over(category_window)
)

product_revenue.orderBy("category", "revenue_rank").show()

+-----------+----------+------------+-------------------+------------+
|   category|product_id|product_name|total_gross_revenue|revenue_rank|
+-----------+----------+------------+-------------------+------------+
|Accessories|       P03|  Headphones|          219614500|           1|
|Electronics|       P01|      Laptop|         3764880000|           1|
|Electronics|       P06|      Tablet|         1882272000|           2|
|Electronics|       P02|     Monitor|         1098104000|           3|
| Networking|       P05|      Router|          282339000|           1|
|     Office|       P04|     Printer|         1333378000|           1|
+-----------+----------+------------+-------------------+------------+



In [59]:
running_window = (
    Window
    .partitionBy("store_id")
    .orderBy("order_date", "order_id")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

running_sales = sales_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
).withColumn(
    "running_gross_revenue",
    F.sum("gross_revenue").over(running_window)
)

running_sales.select(
    "store_id", "order_date", "order_id",
    "gross_revenue", "running_gross_revenue"
).show(20)

+--------+----------+--------+-------------+---------------------+
|store_id|order_date|order_id|gross_revenue|running_gross_revenue|
+--------+----------+--------+-------------+---------------------+
|     S01|2026-01-01|     180|       300000|               300000|
|     S01|2026-01-01|     360|        60000|               360000|
|     S01|2026-01-01|     540|       300000|               660000|
|     S01|2026-01-01|     720|        60000|               720000|
|     S01|2026-01-01|     900|       300000|              1020000|
|     S01|2026-01-01|    1080|        60000|              1080000|
|     S01|2026-01-01|    1260|       300000|              1380000|
|     S01|2026-01-01|    1440|        60000|              1440000|
|     S01|2026-01-01|    1620|       300000|              1740000|
|     S01|2026-01-01|    1800|        60000|              1800000|
|     S01|2026-01-01|    1980|       300000|              2100000|
|     S01|2026-01-01|    2160|        60000|              2160

In [60]:
monthly_revenue = (
    sales_df
    .filter(F.col("order_status") == "Completed")
    .withColumn("month", F.month("order_date"))
    .withColumn(
        "gross_revenue",
        F.col("quantity") * F.col("unit_price")
    )
    .groupBy("month")
    .agg(
        F.sum("gross_revenue").alias("monthly_sales_revenue")
    )
    .orderBy("month")
)

monthly_revenue.show()

+-----+---------------------+
|month|monthly_sales_revenue|
+-----+---------------------+
|    1|           1202895500|
|    2|            966988500|
|    3|           1141462500|
|    4|           1147980500|
|    5|           1202125500|
|    6|           1024744500|
+-----+---------------------+



In [61]:
step1 = sales_df.filter(F.col("quantity") > 5)
print("Transformation 1 created: filter")

step2 = step1.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
)
print("Transformation 2 created: withColumn")

step3 = step2.select(
    "order_id", "product_id", "quantity", "gross_revenue"
)
print("Transformation 3 created: select")

# This action triggers execution
step3.show()

Transformation 1 created: filter
Transformation 2 created: withColumn
Transformation 3 created: select
+--------+----------+--------+-------------+
|order_id|product_id|quantity|gross_revenue|
+--------+----------+--------+-------------+
|      15|       P04|       8|       136000|
|      21|       P04|       6|       102000|
|      39|       P04|       8|       136000|
|      45|       P04|       6|       102000|
|      14|       P03|       7|        24500|
|      38|       P03|       7|        24500|
|      22|       P05|       7|        31500|
|      46|       P05|       7|        31500|
|       5|       P06|       6|       144000|
|      23|       P06|       8|       192000|
|      29|       P06|       6|       144000|
|      47|       P06|       8|       192000|
|      53|       P06|       6|       144000|
|       7|       P02|       8|       112000|
|      13|       P02|       6|        84000|
|      31|       P02|       8|       112000|
|      37|       P02|       6|        8400

In [62]:
aggregated_df = sales_df.groupBy("product_id").agg(
    F.sum("quantity").alias("total_quantity"),
    F.sum(
        F.col("quantity") * F.col("unit_price")
    ).alias("total_gross_revenue")
)

aggregated_df.explain(True)

# collect() brings the results to the driver
results = aggregated_df.collect()
print(results)

== Parsed Logical Plan ==
'Aggregate ['product_id], ['product_id, 'sum('quantity) AS total_quantity#4017, 'sum('`*`('quantity, 'unit_price)) AS total_gross_revenue#4018]
+- Project [order_id#3557L, order_date#3558, store_id#3560, product_id#3559, quantity#3561, unit_price#3566L, discount_pct#3862, transaction_status#3691, salesperson_id#3565, gross_revenue#3656L, CASE WHEN (quantity#3561 <= 3) THEN Small WHEN (quantity#3561 <= 6) THEN Medium ELSE Large END AS order_size#3881]
   +- Filter atleastnnonnulls(1, salesperson_id#3565)
      +- Project [order_id#3557L, order_date#3558, store_id#3560, product_id#3559, quantity#3561, unit_price#3566L, coalesce(discount_pct#3562, cast(0 as int)) AS discount_pct#3862, transaction_status#3691, salesperson_id#3565, gross_revenue#3656L]
         +- Project [order_id#3557L, order_date#3558, store_id#3560, product_id#3559, quantity#3561, unit_price#3566L, discount_pct#3562, transaction_status#3691, salesperson_id#3565, gross_revenue#3656L]
           

In [63]:
completed_df = sales_df.filter(
    F.col("order_status") == "Completed"
).cache()

start = time.time()
count1 = completed_df.count()
time1 = time.time() - start

start = time.time()
count2 = completed_df.count()
time2 = time.time() - start

print("First count:", count1)
print("First execution time:", time1, "seconds")

print("Second count:", count2)
print("Second execution time:", time2, "seconds")

completed_df.unpersist()

First count: 73339
First execution time: 4.702541828155518 seconds
Second count: 73339
Second execution time: 1.6464073657989502 seconds


DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, transaction_status: string, salesperson_id: string, gross_revenue: bigint, order_size: string]

In [64]:
high_quantity_df = sales_df.filter(
    F.col("quantity") > 5
).persist(StorageLevel.MEMORY_AND_DISK)
print("Count:", high_quantity_df.count())
high_quantity_df.show(10)
high_quantity_df.unpersist()

Count: 35295
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|gross_revenue|order_size|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|         Completed|         EMP04|       136000|     Large|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|          Returned|         EMP10|       102000|    Medium|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|         Completed|         EMP04|       136000|     Large|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|         Completed|         EMP10|       102000|    Medium|
|      63|2026-03-05|     S02|       P04|       8

DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, transaction_status: string, salesperson_id: string, gross_revenue: bigint, order_size: string]

In [65]:
high_quantity_df = sales_df.filter(
    F.col("quantity") > 5
).persist(StorageLevel.MEMORY_AND_DISK)
print("Count:", high_quantity_df.count())
high_quantity_df.show(10)
high_quantity_df.unpersist()

Count: 35295
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|gross_revenue|order_size|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+-------------+----------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|         Completed|         EMP04|       136000|     Large|
|      21|2026-01-22|     S04|       P04|       6|     17000|           0|          Returned|         EMP10|       102000|    Medium|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|         Completed|         EMP04|       136000|     Large|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|         Completed|         EMP10|       102000|    Medium|
|      63|2026-03-05|     S02|       P04|       8

DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, transaction_status: string, salesperson_id: string, gross_revenue: bigint, order_size: string]

In [66]:
cached_aggregation = (
    sales_df
    .groupBy("product_id")
    .agg(
        F.sum("quantity").alias("total_quantity"),
        F.sum(
            F.col("quantity") * F.col("unit_price")
        ).alias("total_revenue")
    )
    .cache()
)

# Materialize the cache
cached_aggregation.count()

# Inspect the execution plan
cached_aggregation.explain(True)

cached_aggregation.show()

cached_aggregation.unpersist()

== Parsed Logical Plan ==
'Aggregate ['product_id], ['product_id, 'sum('quantity) AS total_quantity#5869, 'sum('`*`('quantity, 'unit_price)) AS total_revenue#5870]
+- Project [order_id#3557L, order_date#3558, store_id#3560, product_id#3559, quantity#3561, unit_price#3566L, discount_pct#3862, transaction_status#3691, salesperson_id#3565, gross_revenue#3656L, CASE WHEN (quantity#3561 <= 3) THEN Small WHEN (quantity#3561 <= 6) THEN Medium ELSE Large END AS order_size#3881]
   +- Filter atleastnnonnulls(1, salesperson_id#3565)
      +- Project [order_id#3557L, order_date#3558, store_id#3560, product_id#3559, quantity#3561, unit_price#3566L, coalesce(discount_pct#3562, cast(0 as int)) AS discount_pct#3862, transaction_status#3691, salesperson_id#3565, gross_revenue#3656L]
         +- Project [order_id#3557L, order_date#3558, store_id#3560, product_id#3559, quantity#3561, unit_price#3566L, discount_pct#3562, transaction_status#3691, salesperson_id#3565, gross_revenue#3656L]
            +- Pr

DataFrame[product_id: string, total_quantity: bigint, total_revenue: bigint]

In [67]:
print("Existing partitions:", sales_df.rdd.getNumPartitions())

sales_8 = sales_df.repartition(8)

print("Partitions after repartition:", sales_8.rdd.getNumPartitions())

Existing partitions: 2
Partitions after repartition: 8


In [68]:
partition_counts = (
    sales_8
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("record_count")
    )
    .orderBy("partition_id")
)

partition_counts.show()

+------------+------------+
|partition_id|record_count|
+------------+------------+
|           0|       11764|
|           1|       11764|
|           2|       11765|
|           3|       11765|
|           4|       11765|
|           5|       11765|
|           6|       11765|
|           7|       11765|
+------------+------------+



In [69]:
sales_by_store = sales_df.repartition(6, "store_id")

store_partitions = (
    sales_by_store
    .select(
        "store_id",
        F.spark_partition_id().alias("partition_id")
    )
    .distinct()
    .orderBy("partition_id", "store_id")
)

store_partitions.show()

+--------+------------+
|store_id|partition_id|
+--------+------------+
|     S03|           1|
|     S02|           3|
|     S01|           4|
|     S04|           5|
+--------+------------+



In [70]:
sales_12 = sales_df.repartition(12)

print("Initial partitions:", sales_12.rdd.getNumPartitions())

sales_coalesced = sales_12.coalesce(4)
print("After coalesce:", sales_coalesced.rdd.getNumPartitions())

sales_repartitioned = sales_12.repartition(4)
print("After repartition:", sales_repartitioned.rdd.getNumPartitions())

print("Coalesce execution plan:")
sales_coalesced.explain()

print("Repartition execution plan:")
sales_repartitioned.explain()

Initial partitions: 12
After coalesce: 4
After repartition: 4
Coalesce execution plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 4
   +- Coalesce 4
      +- ShuffleQueryStage 3
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=8826]
            +- *(3) Project [order_id#3557L, order_date#3558, store_id#3560, product_id#3559, quantity#3561, unit_price#3566L, coalesce(discount_pct#3562, 0) AS discount_pct#3862, order_status#3563 AS transaction_status#3691, salesperson_id#3565, (cast(quantity#3561 as bigint) * unit_price#3566L) AS gross_revenue#3656L, CASE WHEN (quantity#3561 <= 3) THEN Small WHEN (quantity#3561 <= 6) THEN Medium ELSE Large END AS order_size#3881]
               +- *(3) BroadcastHashJoin [product_id#3559], [product_id#3526], LeftOuter, BuildRight, false
                  :- AQEShuffleRead local
                  :  +- ShuffleQueryStage 0
                  :     +- Exchange hashpartitioning(pr

In [71]:
sales_df.repartition(4).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/sales_output_csv")

print("CSV output folder created.")

CSV output folder created.


In [75]:
completed_sales = sales_df.filter(
    F.col("order_status") == "Completed"
)
completed_sales = completed_sales.na.fill({
    "discount_pct": 0
})
joined_sales = (
    completed_sales
    .join(products_df, "product_id", "inner")
    .join(stores_df, "store_id", "inner")
)
revenue_df = (
    joined_sales
    .withColumn(
        "gross_revenue",
        F.col("quantity") * F.col("list_price")
    )
    .withColumn(
        "discounted_revenue",
        F.col("gross_revenue") *
        (1 - F.col("discount_pct") / 100.0)
    )
)
final_df = (
    revenue_df
    .groupBy("region", "category")
    .agg(
        F.sum("gross_revenue").alias("total_gross_revenue"),
        F.sum("discounted_revenue").alias("total_revenue"),
        F.count("*").alias("transaction_count")
    )
)
region_window = Window.partitionBy("region").orderBy(
    F.desc("total_revenue")
)

final_df = final_df.withColumn(
    "category_rank",
    F.dense_rank().over(region_window)
)
final_df.cache()
final_df.orderBy("region", "category_rank").show(truncate=False)
final_df.repartition(2).write \
    .mode("overwrite") \
    .parquet("/content/final_sales_parquet")

print("Final Parquet output saved.")
final_df.unpersist()

+------+-----------+-------------------+-------------+-----------------+-------------+
|region|category   |total_gross_revenue|total_revenue|transaction_count|category_rank|
+------+-----------+-------------------+-------------+-----------------+-------------+
|North |Electronics|880032000          |8.050656E8   |6111             |1            |
|North |Office     |415446000          |3.878822E8   |6111             |2            |
|North |Networking |137488500          |1.2707145E8  |6111             |3            |
|South |Electronics|2029280000         |1.9051378E9  |18336            |1            |
|South |Office     |623424000          |5.702616E8   |6112             |2            |
|South |Accessories|106960000          |9.8850325E7  |6112             |3            |
|South |Networking |82512000           |7.78212E7    |6112             |4            |
|West  |Electronics|2346896000         |2.164164E9   |12223            |1            |
|West  |Accessories|64158500           |6.0

DataFrame[region: string, category: string, total_gross_revenue: bigint, total_revenue: double, transaction_count: bigint, category_rank: int]

**Pyspark Exercise** **2**

In [90]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [76]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder \
    .appName("LogisticsPracticeSet02") \
    .getOrCreate()
shipments_df = spark.read.option("header", "true").option("inferSchema", "true").csv("shipments_75000.csv")
extra_df = spark.read.option("header", "true").option("inferSchema", "true").csv("shipments_extra_12000.csv")
warehouse_df = spark.read.option("header", "true").option("inferSchema", "true").csv("warehouse_master.csv")
carrier_df = spark.read.option("header", "true").option("inferSchema", "true").csv("carrier_master.csv")
route_df = spark.read.option("header", "true").option("inferSchema", "true").csv("route_master.csv")
sensor_df = spark.read.option("multiline", "true").json("warehouse_sensor_config.json")

print("Setup completed successfully. DataFrames loaded.")

Setup completed successfully. DataFrames loaded.


In [77]:
shipments_df.printSchema()
print(f"Total number of records: {shipments_df.count()}")
shipments_df.show(15, truncate=False)

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Total number of records: 75000
+-----------+------------+-------------+----

In [78]:
selected_shipments = shipments_df.select(
    "shipment_id", "warehouse_id", "service_level", "shipment_status", "shipping_fee_inr"
)
selected_shipments.show(20, truncate=False)

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
|SHP0000001 |W08         |Express      |Delivered      |554.34          |
|SHP0000002 |W08         |Standard     |Delivered      |414.56          |
|SHP0000003 |W01         |Express      |Delivered      |724.64          |
|SHP0000004 |W03         |Standard     |Delivered      |287.86          |
|SHP0000005 |W03         |Economy      |Delivered      |251.68          |
|SHP0000006 |W01         |Express      |Delivered      |477.9           |
|SHP0000007 |W02         |Standard     |In Transit     |298.68          |
|SHP0000008 |W03         |Economy      |In Transit     |284.52          |
|SHP0000009 |W02         |Standard     |Delivered      |861.74          |
|SHP0000010 |W02         |Economy      |Delivered      |309.76          |
|SHP0000011 |W04         |Standard    

In [79]:
filtered_shipments = shipments_df.filter(
    (col("service_level") == "Express") & (col("package_weight_kg") > 5)
)
filtered_shipments.show(20, truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0000003 |2026-01-31  |2026-02-01   |2026-02-03            |2

In [80]:
sorted_shipments = shipments_df.orderBy(col("shipping_fee_inr").desc())
sorted_shipments.show(20, truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0044904 |2026-08-11  |2026-08-13   |2026-08-16            |N

In [81]:
status_counts = shipments_df.groupBy("shipment_status").count().orderBy(col("count").desc())
status_counts.show(truncate=False)

+---------------+-----+
|shipment_status|count|
+---------------+-----+
|Delivered      |52434|
|In Transit     |10472|
|Returned       |6130 |
|Cancelled      |3723 |
|Lost           |2241 |
+---------------+-----+



In [82]:
service_agg = shipments_df.groupBy("service_level").agg(
    sum("shipping_fee_inr").alias("total_shipping_fees"),
    sum(coalesce(col("shipping_cost_inr"), lit(0))).alias("total_shipping_costs")
).withColumn(
    "profit_margin",
    (col("total_shipping_fees") - col("total_shipping_costs")) / col("total_shipping_fees")
)
service_agg.show(truncate=False)

+-------------+--------------------+--------------------+-------------------+
|service_level|total_shipping_fees |total_shipping_costs|profit_margin      |
+-------------+--------------------+--------------------+-------------------+
|Express      |1.0058468059999986E7|7350805.699999983   |0.26919232072403737|
|Standard     |1.3437372619999977E7|1.0110465900000025E7|0.24758610288504138|
|Economy      |4489918.680000002   |3386212.500000007   |0.24581874609809062|
+-------------+--------------------+--------------------+-------------------+



In [83]:
region_shipments = shipments_df.join(warehouse_df, "warehouse_id", "inner") \
    .groupBy("region") \
    .count() \
    .orderBy(col("count").desc())
region_shipments.show(truncate=False)

+------+-----+
|region|count|
+------+-----+
|South |33018|
|West  |26909|
|North |9767 |
|East  |5306 |
+------+-----+



In [84]:
shipments_with_carrier = shipments_df.join(carrier_df, "carrier_id", "inner")
joined_carrier_route = shipments_with_carrier.join(
    route_df,
    "route_id",
    "inner"
)

avg_cost_per_km = joined_carrier_route \
    .filter(col("shipping_cost_inr").isNotNull() & (col("distance_km") > 0)) \
    .withColumn("cost_per_km", col("shipping_cost_inr") / col("distance_km")) \
    .groupBy("carrier_id", "carrier_name") \
    .agg(avg("cost_per_km").alias("avg_shipping_cost_per_km"))

avg_cost_per_km.show(truncate=False)

+----------+--------------------+------------------------+
|carrier_id|carrier_name        |avg_shipping_cost_per_km|
+----------+--------------------+------------------------+
|C07       |BluePeak Delivery   |0.6074832283099014      |
|C04       |AirSprint Couriers  |0.6125755103088726      |
|C06       |RouteWise           |0.5879324060720517      |
|C03       |MetroRoute Logistics|0.594275960971091       |
|C02       |ParcelBridge        |0.5957409571811839      |
|C05       |CargoNest           |0.5897283901696387      |
|C01       |SwiftLine Express   |0.6032748764541479      |
+----------+--------------------+------------------------+



In [85]:
missing_counts = shipments_df.select([
    count(when(col(c).isNull(), c)).alias(f"missing_{c}")
    for c in ["carrier_id", "package_weight_kg", "shipping_cost_inr", "customer_rating"]
])
missing_counts.show(truncate=False)

+------------------+-------------------------+-------------------------+-----------------------+
|missing_carrier_id|missing_package_weight_kg|missing_shipping_cost_inr|missing_customer_rating|
+------------------+-------------------------+-------------------------+-----------------------+
|871               |1129                     |1503                     |26255                  |
+------------------+-------------------------+-------------------------+-----------------------+



In [86]:
df_handling_filled = shipments_df.na.fill(0, subset=["handling_minutes"])
df_valid_carrier = shipments_df.filter(col("carrier_id").isNotNull())

print(f"Original count: {shipments_df.count()}")
print(f"Valid carrier count: {df_valid_carrier.count()}")

Original count: 75000
Valid carrier count: 74129


In [87]:
df_weight_cat = shipments_df.withColumn(
    "weight_category",
    when(col("package_weight_kg").isNull(), "Unknown")
    .when(col("package_weight_kg") < 2, "Light")
    .when((col("package_weight_kg") >= 2) & (col("package_weight_kg") <= 10), "Medium")
    .otherwise("Heavy")
)
df_weight_cat.select("shipment_id", "package_weight_kg", "weight_category").show(20, truncate=False)

+-----------+-----------------+---------------+
|shipment_id|package_weight_kg|weight_category|
+-----------+-----------------+---------------+
|SHP0000001 |3.93             |Medium         |
|SHP0000002 |5.62             |Medium         |
|SHP0000003 |8.78             |Medium         |
|SHP0000004 |0.97             |Light          |
|SHP0000005 |NULL             |Unknown        |
|SHP0000006 |6.05             |Medium         |
|SHP0000007 |1.36             |Light          |
|SHP0000008 |2.04             |Medium         |
|SHP0000009 |18.73            |Heavy          |
|SHP0000010 |2.02             |Medium         |
|SHP0000011 |6.87             |Medium         |
|SHP0000012 |11.83            |Heavy          |
|SHP0000013 |3.73             |Medium         |
|SHP0000014 |1.42             |Light          |
|SHP0000015 |1.74             |Light          |
|SHP0000016 |6.87             |Medium         |
|SHP0000017 |2.58             |Medium         |
|SHP0000018 |1.85             |Light    

In [88]:
df_duration = shipments_df \
    .withColumn("booking_date_dt", to_date(col("booking_date"))) \
    .withColumn("delivery_date_dt", to_date(col("actual_delivery_date"))) \
    .withColumn(
        "delivery_duration_days",
        datediff(col("delivery_date_dt"), col("booking_date_dt"))
    )

df_duration.filter(col("shipment_status") == "Delivered") \
    .select("shipment_id", "booking_date_dt", "delivery_date_dt", "delivery_duration_days") \
    .show(15, truncate=False)

+-----------+---------------+----------------+----------------------+
|shipment_id|booking_date_dt|delivery_date_dt|delivery_duration_days|
+-----------+---------------+----------------+----------------------+
|SHP0000001 |2026-05-24     |2026-05-27      |3                     |
|SHP0000002 |2026-05-17     |2026-05-21      |4                     |
|SHP0000003 |2026-01-31     |2026-02-04      |4                     |
|SHP0000004 |2026-03-12     |2026-03-18      |6                     |
|SHP0000005 |2026-08-16     |2026-08-29      |13                    |
|SHP0000006 |2026-02-06     |2026-02-09      |3                     |
|SHP0000009 |2026-08-12     |2026-08-21      |9                     |
|SHP0000010 |2026-06-01     |2026-06-11      |10                    |
|SHP0000011 |2026-04-24     |2026-05-01      |7                     |
|SHP0000013 |2026-06-26     |2026-06-29      |3                     |
|SHP0000014 |2026-04-07     |2026-04-09      |2                     |
|SHP0000016 |2026-06

In [89]:
df_performance = shipments_df.withColumn(
    "delivery_performance",
    when(col("actual_delivery_date").isNull(), "Missing Actual Date")
    .when(col("actual_delivery_date") < col("promised_delivery_date"), "Early")
    .when(col("actual_delivery_date") == col("promised_delivery_date"), "On Time")
    .otherwise("Late")
)
df_performance.groupBy("delivery_performance").count().show(truncate=False)

+--------------------+-----+
|delivery_performance|count|
+--------------------+-----+
|On Time             |22049|
|Missing Actual Date |22566|
|Late                |21900|
|Early               |8485 |
+--------------------+-----+



In [91]:
df_monthly = shipments_df \
    .filter(col("shipment_status") == "Delivered") \
    .withColumn("month", date_format(to_date(col("booking_date")), "yyyy-MM")) \
    .groupBy("month") \
    .agg(
        count("shipment_id").alias("delivered_shipment_count"),
        sum("shipping_fee_inr").alias("total_shipping_fees")
    ) \
    .orderBy("month")

df_monthly.show(truncate=False)

+-------+------------------------+-------------------+
|month  |delivered_shipment_count|total_shipping_fees|
+-------+------------------------+-------------------+
|2026-01|6710                    |2525138.440000005  |
|2026-02|6097                    |2268113.5000000037 |
|2026-03|6810                    |2530456.2400000026 |
|2026-04|6379                    |2389163.4600000014 |
|2026-05|6858                    |2549683.9999999963 |
|2026-06|6615                    |2447590.1200000006 |
|2026-07|6773                    |2512355.2199999997 |
|2026-08|6192                    |2306432.5199999996 |
+-------+------------------------+-------------------+



In [92]:
from pyspark.sql.window import Window

region_carrier_fees = shipments_df.join(warehouse_df, "warehouse_id") \
    .groupBy("region", "carrier_id") \
    .agg(sum("shipping_fee_inr").alias("total_fees"))

window_region = Window.partitionBy("region").orderBy(col("total_fees").desc())

ranked_carriers = region_carrier_fees.withColumn("rank", dense_rank().over(window_region))
ranked_carriers.show(25, truncate=False)

+------+----------+------------------+----+
|region|carrier_id|total_fees        |rank|
+------+----------+------------------+----+
|East  |C01       |393862.61999999994|1   |
|East  |C02       |373590.02         |2   |
|East  |C03       |302260.92000000004|3   |
|East  |C05       |243322.91999999998|4   |
|East  |C07       |222025.85999999993|5   |
|East  |C04       |211583.64000000004|6   |
|East  |C06       |195810.3399999999 |7   |
|East  |NULL      |24594.6           |8   |
|North |C02       |731442.0999999994 |1   |
|North |C01       |653223.6200000005 |2   |
|North |C03       |597883.4600000005 |3   |
|North |C05       |464601.85999999975|4   |
|North |C07       |425396.69999999995|5   |
|North |C04       |393181.07999999984|6   |
|North |C06       |338296.45999999985|7   |
|North |NULL      |45235.700000000004|8   |
|South |C02       |2442652.5400000014|1   |
|South |C01       |2271284.540000001 |2   |
|South |C03       |1929008.1199999978|3   |
|South |C07       |1519441.49999

In [93]:
window_warehouse = Window.partitionBy("warehouse_id").orderBy(col("booking_date").desc())

latest_shipments = shipments_df \
    .withColumn("rn", row_number().over(window_warehouse)) \
    .filter(col("rn") == 1) \
    .drop("rn")

latest_shipments.select("warehouse_id", "shipment_id", "booking_date").show(truncate=False)

+------------+-----------+------------+
|warehouse_id|shipment_id|booking_date|
+------------+-----------+------------+
|W01         |SHP0000944 |2026-08-28  |
|W02         |SHP0000467 |2026-08-28  |
|W03         |SHP0006120 |2026-08-28  |
|W04         |SHP0000924 |2026-08-28  |
|W05         |SHP0000717 |2026-08-28  |
|W06         |SHP0000410 |2026-08-28  |
|W07         |SHP0000292 |2026-08-28  |
|W08         |SHP0001994 |2026-08-28  |
+------------+-----------+------------+



In [94]:
window_running = Window.partitionBy("warehouse_id") \
    .orderBy("booking_date", "shipment_id") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_total_df = shipments_df.withColumn(
    "running_total_fees",
    sum("shipping_fee_inr").over(window_running)
)

running_total_df.select("warehouse_id", "shipment_id", "booking_date", "shipping_fee_inr", "running_total_fees").show(20, truncate=False)

+------------+-----------+------------+----------------+------------------+
|warehouse_id|shipment_id|booking_date|shipping_fee_inr|running_total_fees|
+------------+-----------+------------+----------------+------------------+
|W01         |SHP0004062 |2026-01-01  |312.48          |312.48            |
|W01         |SHP0006739 |2026-01-01  |227.02          |539.5             |
|W01         |SHP0007553 |2026-01-01  |316.02          |855.52            |
|W01         |SHP0008505 |2026-01-01  |572.68          |1428.1999999999998|
|W01         |SHP0009674 |2026-01-01  |322.9           |1751.1            |
|W01         |SHP0011312 |2026-01-01  |390.1           |2141.2            |
|W01         |SHP0011588 |2026-01-01  |421.72          |2562.92           |
|W01         |SHP0011698 |2026-01-01  |190.22          |2753.14           |
|W01         |SHP0011973 |2026-01-01  |384.64          |3137.7799999999997|
|W01         |SHP0017305 |2026-01-01  |203.48          |3341.2599999999998|
|W01        

In [95]:
combined_shipments_df = shipments_df.unionByName(extra_df)
print(f"Combined total records: {combined_shipments_df.count()}")

Combined total records: 87000


In [96]:
initial_count = combined_shipments_df.count()
deduped_df = combined_shipments_df.dropDuplicates(["shipment_id"])
final_count = deduped_df.count()

print(f"Before deduplication: {initial_count}")
print(f"After deduplication: {final_count}")
print(f"Duplicates removed: {initial_count - final_count}")

Before deduplication: 87000
After deduplication: 86250
Duplicates removed: 750


In [97]:
from pyspark.sql.functions import col, explode

exploded_sensor_df = sensor_df \
    .withColumn("sensor_detail", explode(col("sensors"))) \
    .select(
        "warehouse_id",
        col("sensor_detail.sensor_id").alias("sensor_id"),
        col("sensor_detail.type").alias("sensor_type")
    )

exploded_sensor_df.show(20, truncate=False)

+------------+---------+-----------+
|warehouse_id|sensor_id|sensor_type|
+------------+---------+-----------+
|W01         |W01-S1   |Temperature|
|W01         |W01-S2   |Humidity   |
|W01         |W01-S3   |Door       |
|W02         |W02-S1   |Temperature|
|W02         |W02-S2   |Humidity   |
|W02         |W02-S3   |Door       |
|W03         |W03-S1   |Temperature|
|W03         |W03-S2   |Humidity   |
|W03         |W03-S3   |Door       |
|W04         |W04-S1   |Temperature|
|W04         |W04-S2   |Humidity   |
|W04         |W04-S3   |Door       |
|W05         |W05-S1   |Temperature|
|W05         |W05-S2   |Humidity   |
|W05         |W05-S3   |Door       |
|W06         |W06-S1   |Temperature|
|W06         |W06-S2   |Humidity   |
|W06         |W06-S3   |Door       |
|W07         |W07-S1   |Temperature|
|W07         |W07-S2   |Humidity   |
+------------+---------+-----------+
only showing top 20 rows


In [99]:
pipeline_query = shipments_df \
    .filter(col("service_level") == "Express") \
    .join(warehouse_df, "warehouse_id") \
    .withColumn("fee_adjusted", col("shipping_fee_inr") * 1.05) \
    .groupBy("region") \
    .agg(sum("fee_adjusted").alias("total_adjusted_fees")) \
    .orderBy(col("total_adjusted_fees").desc())

pipeline_query.explain(True)
pipeline_query.show()

== Parsed Logical Plan ==
'Sort ['total_adjusted_fees DESC NULLS LAST], true
+- Aggregate [region#7055], [region#7055, sum(fee_adjusted#9576) AS total_adjusted_fees#9577]
   +- Project [warehouse_id#6983, shipment_id#6978, booking_date#6979, dispatch_date#6980, promised_delivery_date#6981, actual_delivery_date#6982, carrier_id#6984, route_id#6985, customer_segment#6986, service_level#6987, package_weight_kg#6988, shipping_fee_inr#6989, shipping_cost_inr#6990, shipment_status#6991, delivery_attempts#6992, damage_flag#6993, customer_rating#6994, handling_minutes#6995, payment_mode#6996, priority_flag#6997, warehouse_name#7053, city#7054, region#7055, capacity_units#7056, storage_type#7057, ... 1 more fields]
      +- Project [warehouse_id#6983, shipment_id#6978, booking_date#6979, dispatch_date#6980, promised_delivery_date#6981, actual_delivery_date#6982, carrier_id#6984, route_id#6985, customer_segment#6986, service_level#6987, package_weight_kg#6988, shipping_fee_inr#6989, shipping_cos

In [101]:
import time

delivered_df = shipments_df.filter(col("shipment_status") == "Delivered").cache()
start_time = time.time()
print(f"Delivered count: {delivered_df.count()}")
print(f"First action time: {time.time() - start_time:.4f} seconds")
start_time = time.time()
delivered_df.groupBy("service_level").count().show()
print(f"Second action (cached) time: {time.time() - start_time:.4f} seconds")

Delivered count: 52434
First action time: 0.0770 seconds
+-------------+-----+
|service_level|count|
+-------------+-----+
|      Express|14819|
|     Standard|28159|
|      Economy| 9456|
+-------------+-----+

Second action (cached) time: 0.1609 seconds


In [100]:
from pyspark.storagelevel import StorageLevel

express_df = shipments_df.filter(col("service_level") == "Express")
express_df.persist(StorageLevel.MEMORY_AND_DISK)

print(f"Express count: {express_df.count()}")
print(f"Is cached: {express_df.is_cached}")

express_df.groupBy("warehouse_id").count().show(5)
express_df.unpersist()
print(f"Is cached after unpersist: {express_df.is_cached}")

Express count: 21090
Is cached: True
+------------+-----+
|warehouse_id|count|
+------------+-----+
|         W02| 3343|
|         W06| 2334|
|         W04| 2680|
|         W08| 2505|
|         W03| 3153|
+------------+-----+
only showing top 5 rows
Is cached after unpersist: False


In [103]:
repartitioned_df = shipments_df.repartition(8)

partition_counts = repartitioned_df \
    .withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id")

partition_counts.show(truncate=False)

+------------+-----+
|partition_id|count|
+------------+-----+
|0           |9375 |
|1           |9374 |
|2           |9376 |
|3           |9375 |
|4           |9375 |
|5           |9375 |
|6           |9375 |
|7           |9375 |
+------------+-----+



In [102]:
df_rep12 = shipments_df.repartition(12)
df_coalesced = df_rep12.coalesce(3)
df_rep3 = shipments_df.repartition(3)

print("--- Coalesce execution plan ---")
df_coalesced.explain()

print("--- Repartition(3) execution plan ---")
df_rep3.explain()

--- Coalesce execution plan ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=13071]
      +- FileScan csv [shipment_id#6978,booking_date#6979,dispatch_date#6980,promised_delivery_date#6981,actual_delivery_date#6982,warehouse_id#6983,carrier_id#6984,route_id#6985,customer_segment#6986,service_level#6987,package_weight_kg#6988,shipping_fee_inr#6989,shipping_cost_inr#6990,shipment_status#6991,delivery_attempts#6992,damage_flag#6993,customer_rating#6994,handling_minutes#6995,payment_mode#6996,priority_flag#6997] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/shipments_75000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<shipment_id:string,booking_date:date,dispatch_date:date,promised_delivery_date:date,actual...


--- Repartition(3) execution plan ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRob

In [104]:
joined_warehouse = shipments_df.join(warehouse_df, "warehouse_id")

joined_warehouse.repartition(4) \
    .write \
    .mode("overwrite") \
    .parquet("output_shipments_warehouse.parquet")

print("Parquet files written successfully with 4 partitions.")

Parquet files written successfully with 4 partitions.


In [107]:
cleaned_shipments = shipments_df \
    .na.fill(0, subset=["shipping_cost_inr"]) \
    .na.drop(subset=["package_weight_kg"])
pipeline_joined = cleaned_shipments \
    .join(warehouse_df, "warehouse_id", "inner") \
    .join(carrier_df, "carrier_id", "inner") \
    .join(route_df, "route_id", "inner")

pipeline_metrics = pipeline_joined \
    .withColumn("delivery_duration_days", datediff(to_date(col("actual_delivery_date")), to_date(col("booking_date")))) \
    .withColumn("is_late", when(col("actual_delivery_date") > col("promised_delivery_date"), 1).otherwise(0)) \
    .withColumn("shipment_profit", col("shipping_fee_inr") - coalesce(col("shipping_cost_inr"), lit(0)))

final_aggregated = pipeline_metrics.groupBy("region", "carrier_id", "carrier_name").agg(
    sum("shipping_fee_inr").alias("total_revenue"),
    sum("shipment_profit").alias("total_profit"),
    (sum("is_late") / count("shipment_id") * 100).alias("late_delivery_percentage")
)

final_aggregated = final_aggregated.withColumn(
    "on_time_delivery_percentage", lit(100) - col("late_delivery_percentage")
)

window_region_carrier = Window.partitionBy("region").orderBy(col("on_time_delivery_percentage").desc())
final_ranked = final_aggregated.withColumn("carrier_rank_in_region", rank().over(window_region_carrier))
final_ranked.cache()

print(f"Total report rows: {final_ranked.count()}")
final_ranked.show(20, truncate=False)

final_ranked.repartition("region") \
    .write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet("final_logistics_performance_report.parquet")

print("Final logistics performance report saved successfully as partitioned Parquet files.")

Total report rows: 28
+------+----------+--------------------+------------------+------------------+------------------------+---------------------------+----------------------+
|region|carrier_id|carrier_name        |total_revenue     |total_profit      |late_delivery_percentage|on_time_delivery_percentage|carrier_rank_in_region|
+------+----------+--------------------+------------------+------------------+------------------------+---------------------------+----------------------+
|South |C04       |AirSprint Couriers  |1343061.0200000003|343319.41999999975|27.90960451977401       |72.090395480226            |1                     |
|South |C05       |CargoNest           |1411358.360000001 |358015.4600000006 |28.264331210191084      |71.73566878980891          |2                     |
|South |C01       |SwiftLine Express   |2234209.780000001 |564265.3799999983 |28.36120401337793       |71.63879598662207          |3                     |
|South |C06       |RouteWise           |1217186.